In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

In [ ]:
spark = SparkSession.builder \
    .appName("KafkaToIcebergFinal") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "hadoop") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://ton-bucket/iceberg") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .getOrCreate()

In [ ]:
# 2. Définition du schéma
schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("timestamp", StringType(), True),
    StructField("city", StringType(), True),
    StructField("section_id", StringType(), True),
    StructField("direction", StringType(), True),
    StructField("speed", DoubleType(), True),
    StructField("vehicle_count", IntegerType(), True)
])

In [ ]:
# Au lieu de ton code précédent, utilise cette version renforcée
data_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "traffic_events") \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .option("kafka.consumer.poll.timeout.ms", "10000")\
    .load()

In [ ]:
# Si cette cellule échoue, ne va pas plus loin, le problème est la config réseau
# Elle affiche les messages bruts sans Iceberg
df = spark.read \
  .format("kafka") \
  .option("kafka.bootstrap.servers", "kafka:9092") \
  .option("subscribe", "traffic_events_v2") \
  .load()

df.printSchema()

In [ ]:
# Essaie de lister les topics via Spark
# Si cette cellule échoue, c'est que le conteneur Spark ne sait pas qui est "kafka"
topics = spark.read.format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "traffic_events") \
    .option("startingOffsets", "earliest") \
    .option("endingOffsets", "latest") \
    .load()

print("Nombre de messages trouvés :", topics.count())

In [ ]:
# Lecture du vrai topic
data_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "traffic_events") \
    .option("startingOffsets", "earliest") \
    .load()

In [ ]:
# Écriture dans Iceberg
query = data_df.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", "s3a://ton-bucket/checkpoints/traffic_v5") \
    .toTable("my_catalog.default.traffic_events")

query.awaitTermination()

In [ ]:
# Création d'une vue "propre"
df_clean = spark.read.format("iceberg").load("my_catalog.default.traffic_events") \
    .select(from_json(col("value").cast("string"), schema).alias("data")) \
    .select("data.*")

# Affiche pour confirmer que tu as bien les colonnes 'city', 'speed', etc.
df_clean.printSchema()
df_clean.show(5)

In [ ]:
kpi_speed = df_clean.groupBy("city").avg("speed")
kpi_speed.show()

In [ ]:
kpi_density = df_clean.groupBy("section_id").sum("vehicle_count")
kpi_density.show()

In [ ]:
kpi_alerts = df_clean.filter(col("speed") > 90).select("event_id", "city", "speed")
kpi_alerts.show()